# 02. Logistic Regression on Real Data

**Goals for this notebook**

- Work through the full **real-data workflow**, step by step:
  **load → look → split → scale → fit → evaluate → interpret**.
- Use `pandas` to look at the data, and meet two new tools: `train_test_split` for making a test set,
  and `StandardScaler` for putting features on a common scale.
- **Part A** (together): the whole workflow on a dataset of banknotes, genuine vs. forged.
- **Part B** (on your own, then together): the same workflow, guided, on a **medical** dataset,
  diagnosing breast tumors.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Interactive sliders (optional): if ipywidgets is missing, we show a few fixed pictures instead.
try:
    from ipywidgets import interact, FloatSlider, FloatLogSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed: the interactive cells will show fixed pictures instead.")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, precision_score, recall_score)

In [ ]:
# --- exercise checker: just run this cell -----------------------------------------------
import base64 as _b64, json as _json
_ANS = _json.loads(_b64.b64decode("eyJXMSI6IFswLjgsIDAuNjY2NjY2NjY2NjY2NjY2Nl0sICJXMiI6IDAuMiwgIlczIjogMi4wMTM3NTI3MDc0NzA0NzY2LCAiQTEiOiBbMTM3MiwgMC40NDQ2MDY0MTM5OTQxNjkxXSwgIkEzIjogWzk2MCwgNDEyLCAwLjQ0NDc5MTY2NjY2NjY2NjY0LCAwLjQ0NDE3NDc1NzI4MTU1MzRdLCAiQTQiOiAtMC43MDAyODkzNDc4MjkzNzkzLCAiQjEiOiBbMzk4LCAxNzEsIDAuMzcxODU5Mjk2NDgyNDEyMDYsIDAuMzc0MjY5MDA1ODQ3OTUzMl0sICJCMiI6IFswLjAsIDEuMF0sICJCNCI6IFswLjk3MDc2MDIzMzkxODEyODYsIDYwLCAxLCA0LCAxMDZdLCAiQjYiOiBbWyJ3b3JzdCB0ZXh0dXJlIiwgInJhZGl1cyBlcnJvciIsICJ3b3JzdCBzeW1tZXRyeSJdLCBbImNvbXBhY3RuZXNzIGVycm9yIiwgImZyYWN0YWwgZGltZW5zaW9uIGVycm9yIiwgIm1lYW4gY29tcGFjdG5lc3MiXV0sICJCOCI6IFswLjk4NDM3NSwgMC45NTQ1NDU0NTQ1NDU0NTQ2XX0="))

def _has_str(v):
    return isinstance(v, str) or (isinstance(v, (list, tuple)) and any(_has_str(x) for x in v))

def _norm(v):
    return [_norm(x) for x in v] if isinstance(v, (list, tuple, np.ndarray)) else str(v).strip().lower()

def _close(a, b, tol):
    if _has_str(b):
        return _norm(a) == _norm(b)
    try:
        return np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol, rtol=0)
    except Exception:
        return False

def check(key, value, tol=1e-3):
    # Compare your answer for exercise `key` with the expected one.
    if value is None or value is Ellipsis or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, _ANS[key], tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_against(key, value, reference, tol=1e-3):
    # Compare your answer with a reference value computed in the same cell.
    if value is None or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, reference, tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_fn(key, fn, tol=1e-3):
    # Call your function on some test inputs and compare with the expected outputs.
    spec = _ANS[key]
    try:
        outs = [fn(*[np.array(a, dtype=float) for a in args]) for args in spec["args"]]
    except Exception as e:
        print(f"❌ {key}: your function raised an error: {type(e).__name__}: {e}")
        return
    if any(o is None for o in outs):
        print(f"⏳ {key}: not answered yet (your function returns None)")
    elif all(_close(o, ex, tol) for o, ex in zip(outs, spec["out"])):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. Your function gives {outs}; check your work and try again")

## Warm-up: 01b and 01c in three questions

**W1.** A classifier's confusion matrix has TP = 40, FP = 10, FN = 20, TN = 130. What are its precision
and recall? Store them in `w1_answer = [precision, recall]`.

**W2.** A false positive costs 1 and a false negative costs 4. What is the best threshold? (Remember
the formula from 01c.) Store it in `w2_answer`.

**W3.** A feature has weight $w_j = 0.7$. When that feature increases by 1, by what factor are the
**odds** multiplied? Store it in `w3_answer`.

In [ ]:
# YOUR CODE HERE
w1_answer = None
w2_answer = None
w3_answer = None
check("W1", w1_answer)
check("W2", w2_answer)
check("W3", w3_answer)

**Your answer:** *(write here)*

# Part A: banknotes, together

Banks and shops need to detect **forged banknotes**. In this dataset, from the UCI Machine Learning
Repository, each example is a photo of a banknote-like specimen, some genuine and some forged. The
photo wasn't given to us directly. Instead, four numbers were computed from each image (after a
mathematical transformation called a *wavelet transform*):

| feature | roughly... |
|---|---|
| `variance` | how much the pixel values vary |
| `skewness` | how lopsided the distribution of values is |
| `curtosis` | how "peaked" the distribution is (the usual spelling is *kurtosis*) |
| `entropy` | how disordered the image is |

The label is 0 or 1. *A real-data surprise:* the dataset's documentation doesn't say which number
means genuine and which means forged! It's commonly assumed that **1 = forged**, and we'll go with
that, but it's a good reminder that real datasets aren't always well documented.

**Source:** V. Lohweg (2012), *Banknote Authentication*, UCI Machine Learning Repository,
[doi:10.24432/C55P57](https://doi.org/10.24432/C55P57), licensed CC BY 4.0.

## A1. Load and look

Real datasets usually come as tables in files, and we read them with **pandas**. A quick reminder of
the few pandas commands we'll need (a table is a **DataFrame**, like a 2D NumPy array with named
columns):

- `pd.read_csv(path)` reads a table from a file;
- `df.head()` shows the first rows;
- `df.describe()` gives summary statistics of every column;
- `df["column"]` picks a column, and `.values` turns a DataFrame or column into a NumPy array.

In [ ]:
cols = ["variance", "skewness", "curtosis", "entropy", "label"]
bank = pd.read_csv("../data/data_banknote_authentication.txt", header=None, names=cols)
bank.head()

In [ ]:
bank.describe().round(2)

`describe()` shows, for every column: the count, mean, standard deviation (`std`), minimum,
quartiles and maximum. Notice that the four features have **different ranges**: `curtosis` goes up to
about 17, while `entropy` stays between about $-8.5$ and $2.5$. We'll come back to this in A4.

**Your turn (A1).** How many banknotes are in the dataset, and what fraction of them have
label 1? Store the answers in `n_notes` and `frac_forged`.

In [ ]:
# YOUR CODE HERE
n_notes = None
frac_forged = None
print(n_notes, frac_forged)
check("A1", [n_notes, frac_forged])

## A2. Look at the data

With 4 features we can look at pairs of them. Here are two scatter plots, and for each feature a
histogram per class:

In [ ]:
X_bank = bank[["variance", "skewness", "curtosis", "entropy"]].values
y_bank = bank["label"].values
names_bank = ["variance", "skewness", "curtosis", "entropy"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (i, j) in zip(axes, [(0, 1), (2, 3)]):
    ax.scatter(X_bank[y_bank == 0, i], X_bank[y_bank == 0, j], s=10, c="tab:blue", alpha=0.6, label="0 (genuine)")
    ax.scatter(X_bank[y_bank == 1, i], X_bank[y_bank == 1, j], s=10, c="tab:red", alpha=0.6, label="1 (forged)")
    ax.set_xlabel(names_bank[i]); ax.set_ylabel(names_bank[j]); ax.legend()
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for k, ax in enumerate(axes):
    ax.hist(X_bank[y_bank == 0, k], bins=30, alpha=0.6, color="tab:blue", label="0")
    ax.hist(X_bank[y_bank == 1, k], bins=30, alpha=0.6, color="tab:red", label="1")
    ax.set_title(names_bank[k])
axes[0].legend()
plt.show()

**Try it yourself, before reading on:** from the pictures, which single feature separates the two
classes best? Which one is almost useless on its own?

**Your answer:** *(write here)*

## A3. Split into training and test sets

In 01b we generated a fresh test set. With real data we can't: we have **one** dataset. So we split it
ourselves. We **hide** part of the data (the test set), train only on the rest, and at the end we
evaluate on the hidden part. That simulates new, unseen data.

`train_test_split` does this in one line:

- `test_size=0.3` puts 30% of the examples in the test set;
- `stratify=y` keeps the **same class balance** in both parts (so we don't, by bad luck, get a test set
  with almost no forged notes);
- `random_state=0` fixes the randomness, so we all get the same split and the same results.

In [ ]:
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bank, y_bank, test_size=0.3, stratify=y_bank, random_state=0)

print("training set:", X_train_b.shape, "   test set:", X_test_b.shape)

**Your turn (A3).** Check that the split did what we asked: compute the sizes of the two parts and
the fraction of label-1 notes in each. Store them in `a3 = [n_train, n_test, frac_train, frac_test]`.

In [ ]:
# YOUR CODE HERE
a3 = None
print(a3)
check("A3", a3)

## A4. Scale the features

The four features live on different scales. We **standardize** each one: subtract its mean and divide
by its standard deviation:

$$x_j^{\text{scaled}} = \frac{x_j - \text{mean}_j}{\text{std}_j}$$

After this every feature has mean 0 and standard deviation 1. `StandardScaler` does it for us, in two
steps:

- `.fit(X)` **computes** the mean and standard deviation of every column of `X`;
- `.transform(X)` **applies** the formula above, using the stored means and standard deviations.

**Why scale?**

1. **The optimizer.** Logistic regression is trained by a gradient-based optimizer. When one feature is
   measured in thousands and another in tenths, the loss surface is badly stretched in one direction,
   and the optimizer needs many more steps. (We'll see this happen in exercise B7.)
2. **Comparing weights.** As we saw in 01b (exercise 5.B), a weight's size depends on its feature's
   units. After scaling, all features are in the same "units" (standard deviations), so we can compare
   their weights.
3. *(A third reason will appear in notebook 03b.)*

**Important: fit the scaler on the training data only.** The test set plays the role of future data,
which we haven't seen. If we computed the means using the test set too, we'd be quietly using
information from the "future": a small form of cheating called **data leakage**. So we `fit` on the
training data, and then `transform` **both** sets with the training means and standard deviations.

In [ ]:
scaler_b = StandardScaler()
scaler_b.fit(X_train_b)                           # learn means and stds from the TRAINING data only
X_train_b_s = scaler_b.transform(X_train_b)
X_test_b_s = scaler_b.transform(X_test_b)         # same means and stds applied to the test data

print("means learned from training data:", scaler_b.mean_.round(3))
print("stds learned from training data: ", scaler_b.scale_.round(3))
print()
print("scaled training data: means", X_train_b_s.mean(axis=0).round(3), "  stds", X_train_b_s.std(axis=0).round(3))
print("scaled test data:     means", X_test_b_s.mean(axis=0).round(3), "  stds", X_test_b_s.std(axis=0).round(3))

The scaled **training** data has means of exactly 0 and standard deviations of exactly 1. The scaled
**test** data is close to that, but not exactly, which is expected: it was scaled with the training
statistics.

**Your turn (A4).** Standardize the `variance` of the **first** training banknote **by hand**,
using NumPy (`np.mean`, `np.std`) on the training data. Store it in `scaled_by_hand` and compare it
with `X_train_b_s[0, 0]`.

In [ ]:
# YOUR CODE HERE
scaled_by_hand = None
print("by hand:", scaled_by_hand, "   StandardScaler:", X_train_b_s[0, 0])
check("A4", scaled_by_hand)

## A5. Fit and evaluate

In [ ]:
clf_b = LogisticRegression()
clf_b.fit(X_train_b_s, y_train_b)

pred_b = clf_b.predict(X_test_b_s)
proba_b = clf_b.predict_proba(X_test_b_s)[:, 1]

print("test accuracy:", round(accuracy_score(y_test_b, pred_b), 4))
print("test AUC:     ", round(roc_auc_score(y_test_b, proba_b), 4))
print("\nconfusion matrix:\n", confusion_matrix(y_test_b, pred_b))
print()
print(classification_report(y_test_b, pred_b, target_names=["genuine (0)", "forged (1)"]))

An excellent result: almost every test banknote is classified correctly, and the AUC is 1.0. The banknote classes are
**almost linearly separable** in these four features, the ideal situation for a linear model like
logistic regression. Real problems are rarely this easy; Part B is more typical.

## A6. Interpret

The features are standardized, so the weights are comparable.

In [ ]:
coef_b = pd.Series(clf_b.coef_[0], index=names_bank)
print(coef_b.round(3))
print("intercept:", clf_b.intercept_[0].round(3))

coef_b.sort_values().plot.barh(color=["tab:blue" if v < 0 else "tab:red" for v in coef_b.sort_values()])
plt.axvline(0, color="k", linewidth=0.8)
plt.title("Weights (standardized features)")
plt.xlabel("weight: < 0 pushes toward genuine, > 0 toward forged")
plt.show()

**Try it yourself, before reading on:** compare the weights with the histograms in A2. Does the
feature with the largest weight match the feature that looked most useful in the pictures? And what
about `entropy`?

**Your answer:** *(write here)*

## A7. Look at probabilities, not only labels

Which test banknotes is the model **unsure** about?

In [ ]:
unsure = np.where((proba_b > 0.1) & (proba_b < 0.9))[0]
print(f"{len(unsure)} of {len(y_test_b)} test banknotes have a probability between 0.1 and 0.9:\n")
for i in unsure:
    print(f"  banknote {i:3d}: true label {y_test_b[i]},  P(forged) = {proba_b[i]:.3f}")

That's more than you might expect! The AUC is a perfect 1.0: **every** forged note gets a higher
probability than **every** genuine one. Yet 35 notes get probabilities between 0.1 and 0.9, and only 3
are misclassified. Look closely: most of the "unsure" notes are on the **correct** side of 0.5, just
with modest confidence.

The model is more **cautious** than it needs to be. The reason is the extra term sklearn quietly adds
to the loss, **regularization**, which keeps the weights from growing too large, and so keeps the
probabilities away from 0 and 1. That's usually a good thing (notebook 03b explains why), but it means
you shouldn't read a probability of 0.85 too literally.

In practice, the least certain cases are exactly the ones a bank would send to a human expert.

# Part B: breast cancer diagnosis, a guided exercise

Now it's your turn to do the whole workflow, on a **medical** dataset: the **Breast Cancer Wisconsin
(Diagnostic)** dataset. Each example is a breast tumor, described by **30 measurements** computed from
a microscope image of its cells (radius, texture, perimeter, area, smoothness, and so on, each as a
mean, an error, and a "worst" value). The question: is the tumor **malignant** (cancerous) or
**benign**?

The data comes with scikit-learn, so no file is needed.

**A note about the labels.** sklearn codes this dataset as **0 = malignant, 1 = benign**. For us, the
"positive" class, the one we want to detect, is **malignant**. With sklearn's coding, "recall" would
mean "recall of benign tumors", and the signs of all the weights would be flipped relative to what we
want. So we **flip the labels**: `y = 1 - target`, which gives **1 = malignant, 0 = benign**.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_c = cancer.data
y_c = 1 - cancer.target                     # flip: 1 = malignant, 0 = benign
feature_names = list(cancer.feature_names)

print("X shape:", X_c.shape)
print("fraction malignant:", y_c.mean().round(3))
print("first 6 features:", list(feature_names[:6]))

### B1. Split

Split `X_c, y_c` into a training set and a test set: 30% for testing, with `stratify` (so both sets
keep the same class balance), and `random_state=42`. Name the results
`X_train_c, X_test_c, y_train_c, y_test_c`. Then store
`b1 = [n_train, n_test, fraction malignant in training, fraction malignant in test]`.

In [ ]:
# YOUR CODE HERE
b1 = None
print(b1)
check("B1", b1)

### B2. Scale

Scale the features with `StandardScaler`: fit it on the **training data only**, then transform both the
training and the test data. Name the results `X_train_c_s` and `X_test_c_s`.

Check: the scaled training data should have mean 0 and standard deviation 1 in **every** column.
Compute the largest absolute column mean (`max_mean`) and the largest column standard deviation
(`max_std`) of `X_train_c_s`.

*(Why scale? One feature, `mean area`, is measured in the hundreds and thousands, and another,
`mean smoothness`, in hundredths. See the explanation in A4.)*

In [ ]:
# YOUR CODE HERE
max_mean = None
max_std = None
print(max_mean, max_std)
check("B2", [max_mean, max_std], tol=1e-6)

### B3. Fit

Fit a `LogisticRegression` model on the **scaled training data**. Call it `clf_c`.

In [ ]:
# YOUR CODE HERE
clf_c = None
print(clf_c)

### B4. Evaluate on the test set

1. Compute the test **accuracy** and the **confusion matrix**, and print the `classification_report`.
2. Store `b4 = [accuracy, TP, FP, FN, TN]`.
3. In words: how many malignant tumors did the model **miss**? How many benign tumors did it wrongly
   call malignant? Which of these two mistakes is worse here?

In [ ]:
# YOUR CODE HERE
b4 = None
print(b4)
check("B4", b4, tol=0.01)

**Your answer:** *(write here)*

### B5. Look at probabilities, not just labels

For the first 15 test patients, print the true label next to the predicted probability of malignancy.
Then count how many of **all** the test patients have a probability between 0.2 and 0.8, where the
model is unsure. Store the count in `n_unsure`. Are any of the model's mistakes among the unsure
patients?

In [ ]:
# YOUR CODE HERE
n_unsure = None

**Your answer:** *(write here)*

### B6. Interpret the coefficients

Make a table (a pandas `Series`) of the 30 weights, with the feature names as the index, and sort it.

1. Which **3** features push the prediction most strongly toward **malignant**? Store their names in a
   list, `top_malignant`, from the strongest down.
2. Which **3** push most strongly toward **benign**? Store them in `top_benign`, from the strongest
   down.

*A word of caution:* many of these features are strongly **correlated** (radius, perimeter and area
measure almost the same thing). With correlated features, the model can split the credit among them in
unexpected ways, and some weights can even get a surprising sign. Interpret individual weights
carefully.

In [ ]:
# YOUR CODE HERE
top_malignant = None
top_benign = None
print("toward malignant:", top_malignant)
print("toward benign:   ", top_benign)
check("B6", None if top_malignant is None else [top_malignant, top_benign])

### B7 (a twist): what if we don't scale?

Fit a `LogisticRegression` on the **unscaled** training data `X_train_c`, and evaluate it on the
unscaled test data.

1. Compare its test accuracy with the scaled model's.
2. Look at `clf.n_iter_`, the number of optimizer steps used, for both models. sklearn's default limit
   is 100 steps. What happened? (You may also see a `ConvergenceWarning`.)

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

### B8 (a twist): a screening threshold

For screening, missing a malignant tumor is much worse than a false alarm (B4). Compute the test
**recall** and **precision** at threshold **0.2** instead of 0.5. Store them in
`b8 = [recall, precision]`. What did we gain, and what did it cost?

In [ ]:
# YOUR CODE HERE
b8 = None
print("threshold 0.2:", b8)
check("B8", b8, tol=0.01)

**Your answer:** *(write here)*

## Recap

- **The workflow:** load → look → **split** → **scale** → fit → evaluate → interpret.
- **`train_test_split`** holds out a test set that simulates new data. `stratify=y` keeps the class
  balance, and `random_state` makes the split reproducible.
- **`StandardScaler`** makes every feature mean 0 and standard deviation 1.
  - **Fit it on the training data only**; fitting on the test data too is data leakage.
  - Scaling helps the optimizer converge, and makes the weights comparable.
- **Evaluate on the test set only.** Look at the confusion matrix, precision and recall, and not just
  accuracy. Look at the **probabilities** too: the unsure cases are where the mistakes are.
- **Interpret the weights with care:** comparable only after scaling, and tricky when features are
  correlated.
- **Relabel when needed,** so that the positive class (1) is the one you want to detect.
- **Next (03a):** we'll implement logistic regression **ourselves**, from scratch, and check that we get
  the same answers as sklearn.

## References and further reading

**Interactive explanations**

- [Train, Test, and Validation Sets](https://mlu-explain.github.io/train-test-validation/), by
  MLU-Explain: why we hold out data, visually.

**Short readings**

- Google Machine Learning Crash Course,
  [Numerical data: normalization](https://developers.google.com/machine-learning/crash-course/numerical-data/normalization):
  why and how to scale features.

**Documentation (for looking things up)**

- scikit-learn:
  [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html),
  [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html),
  and the [Breast Cancer dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html).
- pandas: [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html), a short
  refresher on DataFrames.

**The datasets**

- Banknote Authentication: V. Lohweg (2012), UCI Machine Learning Repository,
  [doi:10.24432/C55P57](https://doi.org/10.24432/C55P57) (CC BY 4.0).
- Breast Cancer Wisconsin (Diagnostic): W. Wolberg, O. Mangasarian, N. Street and W. Street (1993),
  UCI Machine Learning Repository, [doi:10.24432/C5DW2B](https://doi.org/10.24432/C5DW2B).